# Train Word CNN + Transformer on HuggingFace ASL Data

This notebook:
1. Loads word videos from Google Drive
2. Trains a CNN to classify word frames
3. Extracts CNN features from videos
4. Trains a Transformer on the temporal sequences

**Requirements:** GPU runtime (Runtime → Change runtime type → GPU)

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import random
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Extract Videos from Drive

**Instructions:**
1. Zip your `hf_asl_videos` folder: `zip -r hf_asl_videos.zip hf_asl_videos`
2. Upload `hf_asl_videos.zip` to Google Drive root
3. Run the cell below

In [ ]:
# Extract videos
ZIP_PATH = '/content/drive/MyDrive/hf_asl_videos.zip'

if os.path.exists(ZIP_PATH):
    print('Extracting videos...')
    !unzip -q {ZIP_PATH} -d /content/
    print('Done!')
else:
    print(f'ERROR: {ZIP_PATH} not found!')
    print('Please upload hf_asl_videos.zip to your Google Drive root.')

In [ ]:
# Check videos
VIDEO_ROOT = '/content/hf_asl_videos'
words = sorted([d for d in os.listdir(VIDEO_ROOT) if os.path.isdir(os.path.join(VIDEO_ROOT, d))])
total_videos = sum(len([f for f in os.listdir(os.path.join(VIDEO_ROOT, w)) if f.endswith('.mp4')]) for w in words)
print(f'Found {len(words)} words, {total_videos} videos')

## 3. Extract Frames for CNN Training

In [ ]:
# FIRST: Split videos at the video level (not frame level) to prevent data leakage
FRAMES_ROOT = '/content/word_frames'
FRAMES_PER_VIDEO = 10  # Sample 10 frames per video for CNN training
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1

os.makedirs(FRAMES_ROOT, exist_ok=True)

# Collect all videos per word
all_videos = {}  # word -> list of video filenames
for word in words:
    word_dir = os.path.join(VIDEO_ROOT, word)
    videos = [f for f in os.listdir(word_dir) if f.endswith('.mp4')]
    all_videos[word] = videos

# Split videos into train/val/test PER WORD (stratified split)
random.seed(42)
video_splits = {'train': [], 'val': [], 'test': []}  # list of (word, video_filename)

for word, videos in all_videos.items():
    shuffled = videos.copy()
    random.shuffle(shuffled)
    n = len(shuffled)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)
    
    for vid in shuffled[:n_train]:
        video_splits['train'].append((word, vid))
    for vid in shuffled[n_train:n_train + n_val]:
        video_splits['val'].append((word, vid))
    for vid in shuffled[n_train + n_val:]:
        video_splits['test'].append((word, vid))

print(f"Video splits: Train={len(video_splits['train'])}, Val={len(video_splits['val'])}, Test={len(video_splits['test'])}")

# Extract frames ONLY from train and val videos (CNN never sees test videos!)
train_frame_info = []  # (frame_path, word)
val_frame_info = []

for split in ['train', 'val']:
    frame_list = train_frame_info if split == 'train' else val_frame_info
    
    for word, vid in tqdm(video_splits[split], desc=f'Extracting {split} frames'):
        vid_path = os.path.join(VIDEO_ROOT, word, vid)
        out_word_dir = os.path.join(FRAMES_ROOT, word)
        os.makedirs(out_word_dir, exist_ok=True)
        
        cap = cv2.VideoCapture(vid_path)
        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(frame)
        cap.release()
        
        if len(frames) < FRAMES_PER_VIDEO:
            continue
        
        indices = np.linspace(0, len(frames)-1, FRAMES_PER_VIDEO, dtype=int)
        vid_name = vid.replace('.mp4', '')
        
        for i, idx in enumerate(indices):
            frame_path = os.path.join(out_word_dir, f'{vid_name}_frame{i:02d}.jpg')
            cv2.imwrite(frame_path, frames[idx])
            frame_list.append((frame_path, word))

print(f'\nCNN Train frames: {len(train_frame_info)}')
print(f'CNN Val frames: {len(val_frame_info)}')
print(f'(Test videos held out - CNN never sees them!)')

In [ ]:
# Shuffle frames within each split (but don't mix splits!)
random.shuffle(train_frame_info)
random.shuffle(val_frame_info)

train_frames = train_frame_info
val_frames = val_frame_info

print(f'Train frames: {len(train_frames)}')
print(f'Val frames: {len(val_frames)}')

## 4. Train CNN on Word Frames

In [ ]:
# Dataset
class FrameDataset(Dataset):
    def __init__(self, frame_info, classes, transform):
        self.frame_info = frame_info
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.transform = transform
    
    def __len__(self):
        return len(self.frame_info)
    
    def __getitem__(self, idx):
        path, word = self.frame_info[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), self.class_to_idx[word]

# Transforms with augmentation
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = FrameDataset(train_frames, words, train_transform)
val_dataset = FrameDataset(val_frames, words, val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

print(f'Classes: {len(words)}')
print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')

In [ ]:
# CNN Model
class WordCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = resnet50(weights=ResNet50_Weights.DEFAULT)
        
        # Freeze early layers
        for name, param in self.backbone.named_parameters():
            if 'layer4' not in name and 'fc' not in name:
                param.requires_grad = False
        
        # Replace FC
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)
    
    def get_features(self, x):
        """Extract 512-dim features before final classifier."""
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)
        x = self.backbone.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.backbone.fc[0](x)  # Linear -> 512
        x = self.backbone.fc[1](x)  # ReLU
        return x

model = WordCNN(len(words)).to(device)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# Training
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

NUM_EPOCHS = 20
best_val_acc = 0
best_state = None

print('Training CNN on word frames...')
print('=' * 60)

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}', leave=False):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * len(labels)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += len(labels)
    
    scheduler.step()
    
    # Validate
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += len(labels)
    
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    
    marker = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = model.state_dict().copy()
        marker = ' <- Best!'
    
    print(f'Epoch {epoch+1:2d}: Train={train_acc:.4f}, Val={val_acc:.4f}{marker}')

print('=' * 60)
print(f'Best Val Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.1f}%)')

# Load best model
model.load_state_dict(best_state)

In [ ]:
# Save CNN model
torch.save(best_state, '/content/best_word_cnn.pth')
print('Saved CNN to /content/best_word_cnn.pth')

## 5. Extract CNN Features from Videos

In [ ]:
# Extract 512-dim features from each video
FEATURES_ROOT = '/content/word_cnn_features'
MAX_FRAMES = 60

model.eval()

feature_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def extract_video_features(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(Image.fromarray(frame))
    cap.release()
    
    if not frames:
        return None
    
    # Sample if too many frames
    if len(frames) > MAX_FRAMES:
        indices = np.linspace(0, len(frames)-1, MAX_FRAMES, dtype=int)
        frames = [frames[i] for i in indices]
    
    # Extract features
    batch = torch.stack([feature_transform(f) for f in frames]).to(device)
    with torch.no_grad():
        features = model.get_features(batch).cpu().numpy()
    
    return features

# Process all videos, keeping track of their pre-determined split
all_video_features = {'train': [], 'val': [], 'test': []}  # split -> list of (features, word)

for split in ['train', 'val', 'test']:
    for word, vid in tqdm(video_splits[split], desc=f'Extracting {split} features'):
        vid_path = os.path.join(VIDEO_ROOT, word, vid)
        features = extract_video_features(vid_path)
        if features is not None:
            all_video_features[split].append((features, word))

print(f'\nExtracted features:')
for split in ['train', 'val', 'test']:
    print(f'  {split}: {len(all_video_features[split])} videos')

In [ ]:
# Save features using the pre-determined splits (NO re-shuffling - prevents data leakage!)
for split_name, samples in all_video_features.items():
    split_dir = os.path.join(FEATURES_ROOT, split_name)
    os.makedirs(split_dir, exist_ok=True)
    
    word_counts = {}
    for features, word in samples:
        word_dir = os.path.join(split_dir, word)
        os.makedirs(word_dir, exist_ok=True)
        
        word_counts[word] = word_counts.get(word, 0) + 1
        np.save(os.path.join(word_dir, f'{word}_{word_counts[word]}.npy'), features)
    
    print(f'{split_name}: {len(samples)} videos')

print(f'\nSaved to {FEATURES_ROOT}/')

## 6. Train Transformer on CNN Features

In [ ]:
# Dataset for sequences
class SequenceDataset(Dataset):
    def __init__(self, root, split, classes, max_len=60):
        self.samples = []
        self.max_len = max_len
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        
        split_dir = os.path.join(root, split)
        for word in classes:
            word_dir = os.path.join(split_dir, word)
            if not os.path.isdir(word_dir):
                continue
            for f in os.listdir(word_dir):
                if f.endswith('.npy'):
                    self.samples.append((os.path.join(word_dir, f), word))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, word = self.samples[idx]
        feat = np.load(path).astype(np.float32)
        
        # Pad/truncate
        if len(feat) > self.max_len:
            feat = feat[:self.max_len]
        elif len(feat) < self.max_len:
            pad = np.zeros((self.max_len - len(feat), 512), dtype=np.float32)
            feat = np.vstack([feat, pad])
        
        return torch.tensor(feat), self.class_to_idx[word]

train_seq = SequenceDataset(FEATURES_ROOT, 'train', words)
val_seq = SequenceDataset(FEATURES_ROOT, 'val', words)
test_seq = SequenceDataset(FEATURES_ROOT, 'test', words)

train_seq_loader = DataLoader(train_seq, batch_size=32, shuffle=True)
val_seq_loader = DataLoader(val_seq, batch_size=32)
test_seq_loader = DataLoader(test_seq, batch_size=32)

print(f'Train: {len(train_seq)}, Val: {len(val_seq)}, Test: {len(test_seq)}')

In [ ]:
# Transformer Model
class PoseTransformer(nn.Module):
    def __init__(self, input_dim=512, num_classes=37, d_model=256, nhead=8, num_layers=4, dropout=0.3, max_len=60):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        
        # Positional encoding
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )
    
    def forward(self, x):
        x = self.input_proj(x)
        x = x + self.pe[:, :x.size(1), :]
        x = self.transformer(x)
        x = x.mean(dim=1)  # Global average pooling
        return self.classifier(x)

transformer = PoseTransformer(input_dim=512, num_classes=len(words)).to(device)
print(f'Transformer params: {sum(p.numel() for p in transformer.parameters()):,}')

In [ ]:
# Train Transformer
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(transformer.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

NUM_EPOCHS = 50
best_val_acc = 0
best_transformer_state = None

print('Training Transformer...')
print('=' * 60)

for epoch in range(NUM_EPOCHS):
    # Train
    transformer.train()
    train_correct, train_total = 0, 0
    
    for x, y in train_seq_loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        out = transformer(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        
        train_correct += (out.argmax(1) == y).sum().item()
        train_total += len(y)
    
    scheduler.step()
    
    # Validate
    transformer.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for x, y in val_seq_loader:
            x, y = x.to(device), y.to(device)
            out = transformer(x)
            val_correct += (out.argmax(1) == y).sum().item()
            val_total += len(y)
    
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    
    marker = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_transformer_state = transformer.state_dict().copy()
        marker = ' *'
    
    if (epoch + 1) % 5 == 0 or marker:
        print(f'Epoch {epoch+1:2d}: Train={train_acc:.3f}, Val={val_acc:.3f}{marker}')

print('=' * 60)

# Test
transformer.load_state_dict(best_transformer_state)
transformer.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for x, y in test_seq_loader:
        x, y = x.to(device), y.to(device)
        out = transformer(x)
        test_correct += (out.argmax(1) == y).sum().item()
        test_total += len(y)

test_acc = test_correct / test_total
print(f'Best Val Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.1f}%)')
print(f'Test Accuracy:     {test_acc:.4f} ({test_acc*100:.1f}%)')

## 7. Save Models

In [ ]:
# Save to Drive
SAVE_DIR = '/content/drive/MyDrive/LearningASL_models'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save CNN
torch.save(best_state, os.path.join(SAVE_DIR, 'best_word_cnn.pth'))

# Save Transformer
torch.save(best_transformer_state, os.path.join(SAVE_DIR, 'best_word_transformer.pth'))

# Save class list
with open(os.path.join(SAVE_DIR, 'word_classes.txt'), 'w') as f:
    f.write('\n'.join(words))

print(f'Saved to {SAVE_DIR}:')
print('  - best_word_cnn.pth')
print('  - best_word_transformer.pth')
print('  - word_classes.txt')

In [ ]:
# Download models locally
from google.colab import files

files.download('/content/best_word_cnn.pth')
files.download(os.path.join(SAVE_DIR, 'best_word_transformer.pth'))
files.download(os.path.join(SAVE_DIR, 'word_classes.txt'))